# Optimus 5-Prime: synthetic-data test, real-data reproduction, and benchmark comparison
Sample et al., *Nat Biotech* 2019. PyTorch re-implementation of `pjsample/human_5utr_modeling`.

**Protocol.** The model under test (the Optimus 5-Prime CNN) and three benchmarks:
**BiLSTM**, **LightGBM on hand-made features**, and the paper's **position-specific k-mer linear model**.
Each is trained and tested first on **synthetic data with known rules (Part A)** and then on the **real eGFP polysome data (Part B)**.
Every model uses the same split. MRL is also binned into low/medium/high (training-set tertiles) for accuracy, precision, recall, F1 and confusion matrices.

**Results obtained (CPU; neural nets mean ± sd over 3 seeds):**

### Synthetic (noise ceiling r² = 0.918)

| Model | r² | Spearman | RMSE | MAE | Accuracy | Precision | Recall | F1 |
|---|---|---|---|---|---|---|---|---|
| Optimus 5-Prime CNN | 0.856 ± 0.006 | 0.889 | 0.603 | 0.478 | 0.753 | 0.754 | 0.753 | 0.751 |
| BiLSTM | 0.748 ± 0.063 | 0.847 | 0.793 | 0.621 | 0.711 | 0.712 | 0.711 | 0.711 |
| LightGBM (hand features) | 0.915 | 0.936 | 0.456 | 0.364 | 0.815 | 0.815 | 0.814 | 0.815 |
| k-mer linear | 0.738 | 0.848 | 0.801 | 0.629 | 0.700 | 0.714 | 0.700 | 0.703 |

### Real eGFP (top-20k test)

| Model | r² | Spearman | RMSE | MAE | Accuracy | Precision | Recall | F1 |
|---|---|---|---|---|---|---|---|---|
| Optimus 5-Prime CNN | 0.919 ± 0.006 | 0.942 | 0.547 | 0.397 | 0.869 | 0.822 | 0.822 | 0.819 |
| BiLSTM | 0.866 ± 0.026 | 0.907 | 0.746 | 0.552 | 0.819 | 0.773 | 0.769 | 0.763 |
| LightGBM (hand features) | 0.893 | 0.922 | 0.620 | 0.449 | 0.848 | 0.795 | 0.791 | 0.791 |
| k-mer linear | 0.640 | 0.832 | 1.155 | 0.889 | 0.696 | 0.723 | 0.659 | 0.636 |


Paper reference values: CNN 0.93, k-mer best 0.66 (real data).

Runtime: a Colab GPU is recommended (Runtime > Change runtime type > GPU). On CPU, budget about 2 h for everything.
Put the GEO files in `DATA_DIR`: `GSM3130435_egfp_unmod_1.csv.gz` (required) and `GSM3130443_designed_library.csv.gz` (Part D only).

In [ ]:
!pip -q install h5py ViennaRNA lightgbm
!git clone -q --depth 1 https://github.com/pjsample/human_5utr_modeling.git
import os
DATA_DIR = "/content/drive/MyDrive/optimus_data"   # <- change me
# from google.colab import drive; drive.mount('/content/drive')
EGFP = f"{DATA_DIR}/GSM3130435_egfp_unmod_1.csv.gz"
DESIGNED = f"{DATA_DIR}/GSM3130443_designed_library.csv.gz"
AUTHOR_H5 = "human_5utr_modeling/modeling/saved_models/main_MRL_model.hdf5"
for d in ("data/synth", "results/bench", "figures/bench"): os.makedirs(d, exist_ok=True)

## 0. Code modules
Written to files so the multiprocessing MFE step and the scripts can import them.

**`optimus.py`**: the Optimus 5-Prime CNN (3x Conv1D(120, k=8) > Dense(40) > Dropout 0.2 > 1; Adam 1e-3, batch 128, MSE, 3 epochs), data loading, one-hot encoding, and the Keras-weight loader.

In [ ]:
%%writefile optimus.py
"""Optimus 5-Prime reproduction (Sample et al., Nat Biotech 2019) in PyTorch.

Mirrors pjsample/human_5utr_modeling/modeling/training_MRL_CNN.ipynb.
"""
import numpy as np
import pandas as pd
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
import torch.nn as nn
from scipy import stats

NUC = {"A": 0, "C": 1, "G": 2, "T": 3}


def load_egfp(path, n_keep=280_000, n_test=20_000):
    """Sort by read depth, keep the top n_keep; the top n_test (highest coverage) are the test set."""
    df = pd.read_csv(path, usecols=["utr", "total_reads", "rl"])
    df = df.sort_values("total_reads", ascending=False).reset_index(drop=True).iloc[:n_keep]
    return df.iloc[n_test:].reset_index(drop=True), df.iloc[:n_test].reset_index(drop=True)


def one_hot(seqs, seq_len=50):
    """(N, seq_len, 4) float32. Unknown bases / padding -> all zeros."""
    x = np.zeros((len(seqs), seq_len, 4), dtype=np.float32)
    for i, s in enumerate(seqs):
        for j, c in enumerate(s[:seq_len].upper()):
            k = NUC.get(c)
            if k is not None:
                x[i, j, k] = 1.0
    return x


def r2(obs, pred):
    """Squared Pearson r, as in the paper (scipy linregress r_value**2)."""
    return stats.linregress(obs, pred).rvalue ** 2


class SamePad(nn.Module):
    """Keras 'same' padding for even kernels: pad (k-1)//2 left, the rest right."""
    def __init__(self, k):
        super().__init__()
        self.l, self.r = (k - 1) // 2, k - 1 - (k - 1) // 2

    def forward(self, x):
        return nn.functional.pad(x, (self.l, self.r))


class OptimusCNN(nn.Module):
    """3x Conv1D(120, k=8, same, ReLU) -> Flatten -> Dense(40, ReLU) -> Dropout(0.2) -> Dense(1)."""
    def __init__(self, seq_len=50, n_filters=120, k=8, n_layers=3, hidden=40,
                 dropout_conv=0.0, dropout_dense=0.2):
        super().__init__()
        layers, c_in = [], 4
        for i in range(n_layers):
            layers += [SamePad(k), nn.Conv1d(c_in, n_filters, k), nn.ReLU()]
            if i >= 1:
                layers.append(nn.Dropout(dropout_conv))
            c_in = n_filters
        self.conv = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Linear(seq_len * n_filters, hidden), nn.ReLU(),
                                  nn.Dropout(dropout_dense), nn.Linear(hidden, 1))

    def forward(self, x):                    # x: (N, L, 4), Keras layout
        h = self.conv(x.transpose(1, 2))     # (N, C, L)
        h = h.transpose(1, 2).flatten(1)     # flatten time-major, same order as Keras
        return self.head(h).squeeze(-1)


def train(model, x, y, epochs=3, batch_size=128, lr=1e-3, seed=1337, log_every=500):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999), eps=1e-8)
    model.to(DEVICE)
    X, Y = torch.from_numpy(x), torch.from_numpy(y.astype(np.float32))
    n = len(X)
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n)
        tot = 0.0
        for b, i in enumerate(range(0, n, batch_size)):
            idx = perm[i:i + batch_size]
            loss = nn.functional.mse_loss(model(X[idx].to(DEVICE)), Y[idx].to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(idx)
            if log_every and b % log_every == 0:
                print(f"  epoch {ep+1} batch {b}/{n//batch_size} loss {loss.item():.4f}", flush=True)
        print(f"epoch {ep+1}: train MSE {tot/n:.4f}", flush=True)
    return model


@torch.no_grad()
def predict(model, x, batch_size=4096):
    model.eval(); dev = next(model.parameters()).device
    return np.concatenate([model(torch.from_numpy(x[i:i + batch_size]).to(dev)).cpu().numpy()
                           for i in range(0, len(x), batch_size)])


def load_keras_weights(model, h5_path):
    """Load the authors' trained Keras weights (e.g. main_MRL_model.hdf5) into OptimusCNN."""
    import h5py
    convs = [m for m in model.conv if isinstance(m, nn.Conv1d)]
    denses = [m for m in model.head if isinstance(m, nn.Linear)]
    with h5py.File(h5_path, "r") as f:
        g = f["model_weights"]
        names = sorted([k for k in g.keys() if k.startswith("conv1d")], key=lambda s: int(s.split("_")[-1]))
        dnames = sorted([k for k in g.keys() if k.startswith("dense")], key=lambda s: int(s.split("_")[-1]))

        def get(name):
            grp = g[name]
            while "kernel:0" not in grp:
                grp = grp[list(grp.keys())[0]]
            return np.array(grp["kernel:0"]), np.array(grp["bias:0"])

        for m, nm in zip(convs, names):
            w, b = get(nm)                                   # keras (k, in, out)
            m.weight.data = torch.from_numpy(w.transpose(2, 1, 0).copy())
            m.bias.data = torch.from_numpy(b)
        for m, nm in zip(denses, dnames):
            w, b = get(nm)                                   # keras (in, out)
            m.weight.data = torch.from_numpy(w.T.copy())
            m.bias.data = torch.from_numpy(b)
    return model

**`synth.py`**: the synthetic-data simulator. Its docstring lists the planted rules.

In [ ]:
%%writefile synth.py
"""Synthetic 5'UTR -> MRL simulator with known, biologically motivated rules.

Each 50-nt UTR sits directly upstream of the CDS start codon (position 50 == CDS AUG).
Ground-truth MRL follows a leaky-scanning model:

    MRL_true = 1 + 7 * structure_factor * prod_i (1 - r_i)

over every upstream AUG i, where the repression r_i depends on
  * whether an in-frame stop codon follows, and the frame relative to the CDS:
      - stop codon before the CDS (uORF, reinitiation possible) ......... base 0.40
      - no stop, in frame with CDS (N-terminal extension) ............... base 0.12
      - no stop, out of frame (overlapping oORF) ......................... base 0.75
  * Kozak context: strength = 0.35 + 0.35*[purine at -3] + 0.30*[G at +4]
and the 5' structure factor = exp(0.025 * MFE) (MFE in kcal/mol, <= 0; ViennaRNA).
Observed MRL = MRL_true + N(0, NOISE_SD).

Because MRL_true is known, the best achievable r2 on the noisy labels is
    var(MRL_true) / (var(MRL_true) + NOISE_SD**2).
"""
import numpy as np
from multiprocessing import Pool

STOPS = {"TAA", "TAG", "TGA"}
BASE_REPRESSION = {"inframe": 0.12, "uorf": 0.40, "oorf": 0.75}
NOISE_SD = 0.45
L = 50


def classify_uaug(seq, p):
    """Class of the ATG starting at p in a UTR of length len(seq) that abuts the CDS."""
    for q in range(p + 3, len(seq) - 2, 3):     # any in-frame stop before the CDS -> self-contained uORF
        if seq[q:q + 3] in STOPS:
            return "uorf"
    if (len(seq) - p) % 3 == 0:                  # reads straight into the CDS in frame
        return "inframe"
    return "oorf"                                # out of frame, overlaps the CDS start


def kozak(seq, p):
    m3 = seq[p - 3] if p >= 3 else "A"          # upstream of UTR is the (purine-rich) vector cap region
    p4 = seq[p + 3] if p + 3 < len(seq) else "G"  # if ATG ends the UTR, +4 is the CDS's first base (G of eGFP GTG)
    return 0.35 + 0.35 * (m3 in "AG") + 0.30 * (p4 == "G")


def uaug_list(seq):
    return [p for p in range(len(seq) - 2) if seq[p:p + 3] == "ATG"]


def true_mrl(seq, mfe):
    keep = 1.0
    for p in uaug_list(seq):
        keep *= 1 - BASE_REPRESSION[classify_uaug(seq, p)] * kozak(seq, p)
    return 1 + 7 * np.exp(0.025 * mfe) * keep


def _mfe(s):
    import RNA
    return RNA.fold(s)[1]


def mfe_batch(seqs, procs=2):
    with Pool(procs) as pool:
        return np.array(pool.map(_mfe, seqs, chunksize=2000), dtype=np.float32)


def make_dataset(n, seed=0, procs=2):
    rng = np.random.default_rng(seed)
    seqs = ["".join(s) for s in rng.choice(list("ACGT"), size=(n, L))]
    mfe = mfe_batch(seqs, procs)
    y_true = np.array([true_mrl(s, m) for s, m in zip(seqs, mfe)], dtype=np.float32)
    y_obs = y_true + rng.normal(0, NOISE_SD, n).astype(np.float32)
    return seqs, mfe, y_true, y_obs

**`bench.py`**: the benchmark models (BiLSTM, k-mer features, hand-made features for LightGBM) and the shared metrics.

In [ ]:
%%writefile bench.py
"""Benchmark models + shared evaluation for the Optimus 5-Prime comparison.

Benchmarks
  1. k-mer linear : position-specific k-mer ridge regression (the paper's own baseline)
  2. LightGBM     : gradient-boosted trees on hand-engineered UTR features
                    (uAUG counts by type, Kozak context, stop codons, GC, MFE, 1-3-mer counts)
  3. BiLSTM       : bidirectional LSTM on one-hot sequence (a different deep-learning family)
Model under test
  Optimus 5-Prime CNN (optimus.py)
"""
import itertools
from collections import Counter
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
from scipy import stats
from sklearn.metrics import (accuracy_score, confusion_matrix, mean_absolute_error,
                             mean_squared_error, precision_recall_fscore_support)

STOPS = ("TAA", "TAG", "TGA")


# ----------------------------------------------------------------------------- metrics
def tertile_edges(y_train):
    """Class boundaries (low / medium / high MRL) from the TRAINING labels only."""
    return np.quantile(y_train, [1 / 3, 2 / 3])


def to_class(y, edges):
    return np.digitize(y, edges)          # 0 = low, 1 = medium, 2 = high


def evaluate(y_obs, y_pred, edges):
    c_obs, c_pred = to_class(y_obs, edges), to_class(y_pred, edges)
    p, r, f, _ = precision_recall_fscore_support(c_obs, c_pred, average="macro", zero_division=0)
    return {
        "r2": float(stats.pearsonr(y_obs, y_pred)[0] ** 2),
        "spearman": float(stats.spearmanr(y_obs, y_pred)[0]),
        "rmse": float(np.sqrt(mean_squared_error(y_obs, y_pred))),
        "mae": float(mean_absolute_error(y_obs, y_pred)),
        "accuracy": float(accuracy_score(c_obs, c_pred)),
        "precision_macro": float(p), "recall_macro": float(r), "f1_macro": float(f),
        "confusion_matrix": confusion_matrix(c_obs, c_pred, labels=[0, 1, 2]).tolist(),
    }


# ----------------------------------------------------------------------------- BiLSTM
class BiLSTM(nn.Module):
    """One-hot (N, L, 4) -> BiLSTM(64) -> [mean-pool ; max-pool] -> Dense(40, ReLU) -> Dropout 0.2 -> 1."""
    def __init__(self, hidden=64, dense=40, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(4, hidden, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(nn.Linear(4 * hidden, dense), nn.ReLU(), nn.Dropout(dropout), nn.Linear(dense, 1))

    def forward(self, x):
        h, _ = self.lstm(x)
        return self.head(torch.cat([h.mean(1), h.amax(1)], dim=1)).squeeze(-1)


# ----------------------------------------------------------------------------- k-mer linear
def kmer_features(seqs, k, L=50):
    idx = {"".join(p): i for i, p in enumerate(itertools.product("ACGT", repeat=k))}
    V, P = len(idx), L - k + 1
    rows, cols = [], []
    for r, s in enumerate(seqs):
        for p in range(P):
            j = idx.get(s[p:p + k])
            if j is not None:
                rows.append(r); cols.append(p * V + j)
    return sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(len(seqs), P * V))


# ----------------------------------------------------------------------------- hand-made features
_KMERS = ["".join(p) for k in (1, 2, 3) for p in itertools.product("ACGT", repeat=k)]


def utr_features(seq, mfe):
    """Interpretable features a biologist would compute by hand (no position-specific one-hot)."""
    n = len(seq)
    f = {"mfe": mfe, "gc": (seq.count("G") + seq.count("C")) / n}
    atgs = [p for p in range(n - 2) if seq[p:p + 3] == "ATG"]
    cnt = {"inframe": 0, "uorf": 0, "oorf": 0}
    best_kozak = {"inframe": 0.0, "uorf": 0.0, "oorf": 0.0}
    for p in atgs:
        has_stop = any(seq[q:q + 3] in STOPS for q in range(p + 3, n - 2, 3))
        kind = "uorf" if has_stop else ("inframe" if (n - p) % 3 == 0 else "oorf")
        cnt[kind] += 1
        m3 = seq[p - 3] if p >= 3 else "A"
        p4 = seq[p + 3] if p + 3 < n else "G"
        best_kozak[kind] = max(best_kozak[kind], (m3 in "AG") + (p4 == "G"))
    for k in cnt:
        f[f"n_uaug_{k}"] = cnt[k]; f[f"kozak_{k}"] = best_kozak[k]
    f["n_uaug"] = len(atgs)
    f["first_uaug_dist"] = (n - atgs[0]) if atgs else 0
    f["last_uaug_dist"] = (n - atgs[-1]) if atgs else 0
    for fr in range(3):
        f[f"n_stop_frame{fr}"] = sum(seq[q:q + 3] in STOPS for q in range(fr, n - 2, 3))
    c = Counter(seq[i:i + k] for k in (1, 2, 3) for i in range(n - k + 1))
    for km in _KMERS:
        f[f"cnt_{km}"] = c.get(km, 0)
    return f


def feature_matrix(seqs, mfes):
    import pandas as pd
    return pd.DataFrame([utr_features(s, m) for s, m in zip(seqs, mfes)])

**`runner.py`**: trains one model on one dataset under the shared protocol and saves its metrics and predictions.

In [ ]:
%%writefile runner.py
"""Train one model on one dataset and evaluate it on that dataset's 20k test set.

usage: python3 runner.py {synth|real} {cnn|bilstm|lgbm|kmer} [seed]

Same protocol for every model:
  * identical train/test split (real: paper's top-20k-by-reads test set; synth: random 20k)
  * last 10% of the shuffled training set = validation, used ONLY by the classical models
    for hyperparameter choice / early stopping; neural nets use the paper's fixed 3-epoch recipe
  * MRL standardised with training-set mean/sd for the neural nets
  * low/medium/high classes for the confusion matrix = tertiles of the training MRL
"""
import json, sys, time, os
import numpy as np, pandas as pd, torch
from optimus import OptimusCNN, train, predict, one_hot
from bench import BiLSTM, evaluate, tertile_edges, kmer_features, feature_matrix

def run_model(dataset, model_name, seed=1337):
    root = "data/synth" if dataset == "synth" else "data"
    out = f"results/bench/{dataset}_{model_name}" + (f"_s{seed}" if model_name in ("cnn", "bilstm") else "")
    os.makedirs("results/bench", exist_ok=True)

    tr, te = pd.read_csv(f"{root}/train.csv"), pd.read_csv(f"{root}/test.csv")
    ytr, yte = tr.rl.values.astype(np.float32), te.rl.values.astype(np.float32)
    edges = tertile_edges(ytr)
    rng = np.random.default_rng(0)
    perm = rng.permutation(len(tr)); n_val = len(tr) // 10
    i_fit, i_val = perm[n_val:], perm[:n_val]
    t0 = time.time(); extra = {}

    if model_name in ("cnn", "bilstm"):
        xtr, xte = one_hot(tr.utr), one_hot(te.utr)
        mu, sd = ytr.mean(), ytr.std()
        net = OptimusCNN() if model_name == "cnn" else BiLSTM()
        net = train(net, xtr, (ytr - mu) / sd, epochs=3, seed=seed, log_every=0)
        pred = predict(net, xte) * sd + mu
        torch.save(net.state_dict(), out + ".pt")
        extra["n_params"] = int(sum(p.numel() for p in net.parameters()))

    elif model_name == "kmer":
        from sklearn.linear_model import Ridge
        best = None
        for k in (3, 4, 5, 6):
            X = kmer_features(tr.utr, k)
            for a in (1, 10, 100, 1000):
                m = Ridge(alpha=a, solver="sparse_cg").fit(X[i_fit], ytr[i_fit])
                v = np.corrcoef(ytr[i_val], m.predict(X[i_val]))[0, 1] ** 2
                print(f"k={k} alpha={a} val r2={v:.4f}", flush=True)
                if best is None or v > best[2]:
                    best = (k, a, v)
        k, a, _ = best
        m = Ridge(alpha=a, solver="sparse_cg").fit(kmer_features(tr.utr, k), ytr)
        pred = m.predict(kmer_features(te.utr, k))
        extra.update(k=k, alpha=a)

    elif model_name == "lgbm":
        import lightgbm as lgb
        Ftr, Fte = feature_matrix(tr.utr, tr.mfe), feature_matrix(te.utr, te.mfe)
        m = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.05, num_leaves=63, min_child_samples=50,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=seed, verbose=-1)
        m.fit(Ftr.iloc[i_fit], ytr[i_fit], eval_set=[(Ftr.iloc[i_val], ytr[i_val])],
              callbacks=[lgb.early_stopping(100, verbose=False)])
        pred = m.predict(Fte)
        imp = pd.Series(m.booster_.feature_importance("gain"), index=Ftr.columns).sort_values(ascending=False)
        extra.update(best_iter=int(m.best_iteration_), n_features=Ftr.shape[1],
                     top_features=(imp.head(10) / imp.sum()).round(4).to_dict())

    res = {"dataset": dataset, "model": model_name, "seed": seed, **evaluate(yte, pred, edges),
           "minutes": round((time.time() - t0) / 60, 1), **extra}
    if dataset == "synth":   # how close to the noise ceiling, and how well the hidden true function is recovered
        res["r2_vs_true_mrl"] = float(np.corrcoef(te.rl_true, pred)[0, 1] ** 2)
    np.save(out + "_pred.npy", pred)
    json.dump(res, open(out + ".json", "w"), indent=1)
    print({k: v for k, v in res.items() if k != "confusion_matrix"})
    return res


if __name__ == "__main__":
    run_model(sys.argv[1], sys.argv[2], int(sys.argv[3]) if len(sys.argv) > 3 else 1337)

In [ ]:
from runner import run_model
from optimus import *
import json, numpy as np, pandas as pd

# Part A: synthetic data (known ground truth)
280,000 random 50-nt UTRs. The true MRL comes from planted rules: uORF, overlapping-ORF and in-frame uAUG penalties scaled by Kozak strength, plus a folding-energy penalty. Gaussian noise (sd 0.45) is added on top.
Because the true MRL is known, the **noise ceiling** (the best r² any model can reach) can be computed, and we can check *which rules* each model learned.

In [ ]:
from synth import make_dataset, NOISE_SD
seqs, mfe, yt, yo = make_dataset(280_000, seed=0)
df = pd.DataFrame({"utr": seqs, "mfe": mfe, "rl_true": yt, "rl": yo})
df.iloc[20_000:].to_csv("data/synth/train.csv", index=False); df.iloc[:20_000].to_csv("data/synth/test.csv", index=False)
te_s = df.iloc[:20_000]
print("MRL mean %.2f sd %.2f | noise ceiling r2 = %.3f" % (df.rl.mean(), df.rl.std(),
      np.var(te_s.rl_true) / (np.var(te_s.rl_true) + NOISE_SD**2)))

### A1. Train and test all four models on synthetic data

In [ ]:
SEEDS = [1337, 1, 2]
for m in ["cnn", "bilstm"]:
    for s in SEEDS:
        run_model("synth", m, s)
run_model("synth", "lgbm"); run_model("synth", "kmer");

### A2. Did the networks learn the planted rules?
An ATG is inserted at every position of 400 uAUG-free test UTRs, and each model's predicted change in MRL is compared with the true change from the simulator.

In [ ]:
%%writefile 13_synth_rule_recovery.py
"""Did the networks learn the rules we planted? In-silico ATG insertion on the synthetic data.

Take 400 test UTRs with no uAUG, write 'ATG' at every position 0..47, and compare
  true change in MRL (simulator, MFE recomputed)   vs   predicted change (model).
Also report each model's mean prediction for single-uAUG test UTRs per planted class.
"""
import json, numpy as np, pandas as pd, torch
from synth import true_mrl, mfe_batch, uaug_list, classify_uaug
from optimus import OptimusCNN, predict, one_hot
from bench import BiLSTM

tr, te = pd.read_csv("data/synth/train.csv"), pd.read_csv("data/synth/test.csv")
mu, sd = tr.rl.mean(), tr.rl.std()
models = {}
for name, cls in (("cnn", OptimusCNN), ("bilstm", BiLSTM)):
    m = cls(); m.load_state_dict(torch.load(map_location='cpu', f=f"results/bench/synth_{name}_s1337.pt")); models[name] = m

base = te[~te.utr.str.contains("ATG")].sample(400, random_state=0).reset_index(drop=True)
rows, mut = [], []
for i, s in enumerate(base.utr):
    for p in range(48):
        m = s[:p] + "ATG" + s[p + 3:]
        if m.count("ATG") == 1:              # exactly one uAUG after insertion
            rows.append((i, p, m)); mut.append(m)
mfe_mut = mfe_batch(mut)
df = pd.DataFrame(rows, columns=["i", "pos", "utr"])
df["dist"] = 50 - df.pos
df["kind"] = [classify_uaug(s, uaug_list(s)[0]) for s in df.utr]
df["true_delta"] = [true_mrl(s, f) for s, f in zip(df.utr, mfe_mut)] - \
                   np.array([true_mrl(base.utr[i], base.mfe[i]) for i in df.i])
x_base, x_mut = one_hot(base.utr), one_hot(df.utr)
out = {}
for name, m in models.items():
    pb = predict(m, x_base) * sd + mu
    df[f"{name}_delta"] = predict(m, x_mut) * sd + mu - pb[df.i]
    out[name] = {"r_true_vs_pred_delta": float(np.corrcoef(df.true_delta, df[f"{name}_delta"])[0, 1]),
                 "mean_delta_by_kind": df.groupby("kind")[f"{name}_delta"].mean().round(3).to_dict()}
out["true"] = {"mean_delta_by_kind": df.groupby("kind").true_delta.mean().round(3).to_dict(),
               "n_by_kind": df.kind.value_counts().to_dict()}
df.to_csv("results/bench/synth_ism.csv", index=False)
json.dump(out, open("results/bench/synth_ism.json", "w"), indent=1)
print(json.dumps(out, indent=1))

In [ ]:
!python 13_synth_rule_recovery.py

In [ ]:
te_s = pd.read_csv("data/synth/test.csv"); p = np.load("results/bench/synth_cnn_s1337_pred.npy")
no = ~te_s.utr.str.contains("ATG")
print("CNN r2 vs TRUE MRL | with uAUG: %.3f   no uAUG (only folding energy varies): %.3f" % (
      np.corrcoef(p[~no], te_s.rl_true[~no])[0,1]**2, np.corrcoef(p[no], te_s.rl_true[no])[0,1]**2))

# Part B: real eGFP polysome data (GSE114002, unmodified rep 1)
Same split as the paper: the top 280k UTRs by read depth, of which the top 20k form the test set. MFE is computed for the LightGBM features.

In [ ]:
tr, te = load_egfp(EGFP)
from synth import mfe_batch
tr["mfe"], te["mfe"] = mfe_batch(list(tr.utr)), mfe_batch(list(te.utr))
tr.to_csv("data/train.csv", index=False); te.to_csv("data/test.csv", index=False)
print(len(tr), len(te))

### B1. Pipeline check: the authors' trained weights should give r² ≈ 0.93

In [ ]:
m = load_keras_weights(OptimusCNN(), AUTHOR_H5)
print("authors' model test r2:", r2(te.rl.values, predict(m, one_hot(te.utr))))

### B2. Train and test all four models on real data

In [ ]:
for m in ["cnn", "bilstm"]:
    for s in SEEDS:
        run_model("real", m, s)
run_model("real", "lgbm"); run_model("real", "kmer");

# Part C: comparison (tables, r² chart, confusion matrices, rule-recovery plot)

In [ ]:
%%writefile 14_bench_figures.py
import json, glob, os, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from synth import NOISE_SD

INK, MUTED, GRID = "#1f1f1e", "#73726c", "#e4e3dd"
MODELS = [("cnn", "Optimus 5-Prime CNN", "#2a78d6"), ("bilstm", "BiLSTM", "#eb6834"),
          ("lgbm", "LightGBM (hand features)", "#1baf7a"), ("kmer", "k-mer linear", "#eda100")]
plt.rcParams.update({"font.size": 10.5, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.titlesize": 11.5, "axes.titleweight": "bold", "savefig.dpi": 200, "savefig.bbox": "tight"})
os.makedirs("figures/bench", exist_ok=True)


def load(ds, m):
    runs = [json.load(open(f)) for f in sorted(glob.glob(f"results/bench/{ds}_{m}*.json")) if not f.endswith("ism.json")]
    return runs


te_s = pd.read_csv("data/synth/test.csv")
ceiling = np.var(te_s.rl_true) / (np.var(te_s.rl_true) + NOISE_SD ** 2)

# ---------------------------------------------------------------- summary table
rows = []
for ds in ("synth", "real"):
    for m, label, _ in MODELS:
        runs = load(ds, m)
        if not runs: continue
        agg = {k: np.mean([r[k] for r in runs]) for k in ("r2", "spearman", "rmse", "mae", "accuracy",
                                                           "precision_macro", "recall_macro", "f1_macro")}
        agg["r2_sd"] = np.std([r["r2"] for r in runs]) if len(runs) > 1 else np.nan
        agg["r2_vs_true_mrl"] = np.mean([r.get("r2_vs_true_mrl", np.nan) for r in runs])
        rows.append({"dataset": ds, "model": label, "n_runs": len(runs), **agg})
summ = pd.DataFrame(rows)
summ.to_csv("results/bench/summary.csv", index=False)
with open("results/bench/summary.md", "w") as f:
    for ds, title in (("synth", f"Synthetic (noise ceiling r² = {ceiling:.3f})"), ("real", "Real eGFP (top-20k test)")):
        f.write(f"### {title}\n\n| Model | r² | Spearman | RMSE | MAE | Accuracy | Precision | Recall | F1 |\n|---|---|---|---|---|---|---|---|---|\n")
        for _, r in summ[summ.dataset == ds].iterrows():
            r2s = f"{r.r2:.3f}" + (f" ± {r.r2_sd:.3f}" if r.n_runs > 1 else "")
            f.write(f"| {r.model} | {r2s} | {r.spearman:.3f} | {r.rmse:.3f} | {r.mae:.3f} | {r.accuracy:.3f} | "
                    f"{r.precision_macro:.3f} | {r.recall_macro:.3f} | {r.f1_macro:.3f} |\n")
        f.write("\n")
print(open("results/bench/summary.md").read())

# ---------------------------------------------------------------- r2 small multiples
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, ds, title in zip(axes, ("synth", "real"), ("Synthetic data (known rules)", "Real eGFP polysome data")):
    present = [(m, l, c) for m, l, c in MODELS if load(ds, m)]
    y = np.arange(len(MODELS))[::-1]
    for yi, (m, l, c) in zip(y, MODELS):
        runs = load(ds, m)
        if not runs: continue
        v = [r["r2"] for r in runs]
        ax.barh(yi, np.mean(v), color=c, height=0.6)
        if len(v) > 1:
            ax.errorbar(np.mean(v), yi, xerr=np.std(v), color=INK, capsize=3, lw=1)
        ax.text(np.mean(v) + 0.015 + (np.std(v) if len(v) > 1 else 0), yi, f"{np.mean(v):.3f}", va="center", color=INK, size=9.5,
                bbox=dict(fc="white", ec="none", pad=1.5), zorder=5)
    if ds == "synth":
        ax.axvline(ceiling, color=MUTED, ls="--", lw=1)
        ax.text(ceiling - 0.01, -0.75, f"noise ceiling {ceiling:.3f}", color=MUTED, size=8.5, ha="right")
    ax.set_yticks(y, [l for _, l, _ in MODELS]); ax.set_xlim(0, 1.08); ax.set_ylim(-1, len(MODELS) - 0.4)
    ax.set_xlabel("Test r² (n = 20,000)"); ax.set_title(title, loc="left")
    ax.xaxis.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
fig.savefig("figures/bench/r2_synth_vs_real.png"); plt.close(fig)

# ---------------------------------------------------------------- confusion matrices
fig, axes = plt.subplots(2, 4, figsize=(12, 6.2))
labels = ["low", "med", "high"]
for i, ds in enumerate(("synth", "real")):
    for j, (m, l, _) in enumerate(MODELS):
        ax = axes[i, j]; runs = load(ds, m)
        if not runs: ax.axis("off"); continue
        r0 = next((r for r in runs if r["seed"] == 1337), runs[0])
        cm = np.array(r0["confusion_matrix"], float)
        cmn = cm / cm.sum(1, keepdims=True)
        ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
        for a in range(3):
            for b in range(3):
                ax.text(b, a, f"{cmn[a, b]:.2f}\n({int(cm[a, b]):,})", ha="center", va="center", size=8,
                        color="white" if cmn[a, b] > 0.6 else INK)
        ax.set_xticks(range(3), labels); ax.set_yticks(range(3), labels)
        ax.set_title(f"{l}\nF1 {r0['f1_macro']:.3f}", size=9.5)
        if j == 0: ax.set_ylabel(("Synthetic" if ds == "synth" else "Real eGFP") + "\nobserved class")
        if i == 1: ax.set_xlabel("predicted class")
        for s in ax.spines.values(): s.set_visible(False)
fig.suptitle("Confusion matrices (seed 1337): MRL binned into training-set tertiles, row-normalised", x=0.02, ha="left",
             fontweight="bold", size=11.5)
fig.tight_layout(); fig.savefig("figures/bench/confusion_matrices.png"); plt.close(fig)

# ---------------------------------------------------------------- ATG insertion (rule recovery)
if os.path.exists("results/bench/synth_ism.csv"):
    d = pd.read_csv("results/bench/synth_ism.csv")
    g = d.groupby("dist")[["true_delta", "cnn_delta", "bilstm_delta"]].mean()
    ism = json.load(open("results/bench/synth_ism.json"))
    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.axhline(0, color=GRID, lw=1)
    ax.plot(g.index, g.true_delta, color=INK, lw=2, marker="o", ms=4, label="ground truth (simulator)")
    ax.plot(g.index, g.cnn_delta, color="#2a78d6", lw=2, label=f"CNN  (r = {ism['cnn']['r_true_vs_pred_delta']:.2f})")
    ax.plot(g.index, g.bilstm_delta, color="#eb6834", lw=2, label=f"BiLSTM  (r = {ism['bilstm']['r_true_vs_pred_delta']:.2f})")
    ax.invert_xaxis()
    ax.set(xlabel="Inserted ATG position (nt upstream of CDS start)", ylabel="Mean change in MRL")
    ax.set_title("Synthetic: effect of inserting one uAUG, true vs learned", loc="left")
    ax.legend(frameon=False, loc="lower left", ncols=3, bbox_to_anchor=(0, -0.42))
    fig.savefig("figures/bench/synth_atg_insertion.png"); plt.close(fig)
print("figures done")

In [ ]:
!python 14_bench_figures.py

In [ ]:
from IPython.display import Image, display, Markdown
display(Markdown(open("results/bench/summary.md").read()))
for p in ["r2_synth_vs_real", "confusion_matrices", "synth_atg_insertion"]:
    display(Image(f"figures/bench/{p}.png", width=900))

# Part D (optional): human 5'UTRs + ClinVar SNVs
Paper Fig 4a reports 0.81 using a *retrained* model whose weights and exact recipe are not in the repo. The original model gives about 0.78–0.79.

In [ ]:
df = pd.read_csv(DESIGNED, low_memory=False)
hu = df[df.library.isin(["human_utrs", "snv"])].sort_values("total", ascending=False).iloc[:25000]
x = one_hot(hu.utr); ytr = pd.read_csv("data/train.csv").rl.values; mu, sd = ytr.mean(), ytr.std()
print("authors' original weights:", r2(hu.rl, predict(load_keras_weights(OptimusCNN(), AUTHOR_H5), x)))
for s in SEEDS:
    m = OptimusCNN(); m.load_state_dict(torch.load(f"results/bench/real_cnn_s{s}.pt", map_location="cpu"))
    print("our CNN seed", s, r2(hu.rl, predict(m, x) * sd + mu))